# Laboratorio Final — Starbucks Rewards: del experimento a la decisión
### Versión Databricks (Unity Catalog) — resuelta

Cierra el bloque de Churn y Uplift Modeling. Trabajamos con la campaña COMPLETA:
**40,000 clientes**, mitad recibió un cupón de $5 (tratamiento), mitad no (control), medidos a 60 días.

| Dónde lo vieron | Qué repetimos aquí |
|---|---|
| Breakout 2 — Las dos ventanas | Modelo de riesgo/churn para targetear y ver la pérdida |
| Breakout 3 — Los 4 cuadrantes | El T-learner: dos modelos, uno por grupo |
| Breakout 4 — La curva Qini | Qué tan bien ordena el T-learner con datos que nunca vio |

### Datos (ya cargados en tu workspace)
Los CSV ya están subidos al volumen de Unity Catalog **`workspace.default.semana3`**:
- `uplift_campaign.csv` (base para entrenar)
- `uplift_campaign_FULL.csv` (base con el tipo real — la revelación final)

La siguiente celda apunta a `/Volumes/workspace/default/semana3`. Si los subes a otra carpeta,
cámbiala en el widget `base_path`. Para volver a subirlos manualmente:

```bash
databricks fs cp data/uplift_campaign.csv      dbfs:/Volumes/workspace/default/semana3/uplift_campaign.csv --overwrite
databricks fs cp data/uplift_campaign_FULL.csv dbfs:/Volumes/workspace/default/semana3/uplift_campaign_FULL.csv --overwrite
```

> El widget permite cambiar la ruta sin editar el código.

In [0]:
# ============================================================
# Configuración de rutas — Databricks (Unity Catalog Volume)
# ============================================================
import os

# Databricks: volumen workspace.default.semana3 (los CSV ya están ahí).
# Fuera de Databricks: usa "data" y los lee del disco local.
try:
    dbutils.widgets.text("base_path", "/Volumes/workspace/default/semana3", "Carpeta con los CSV")
    dbutils.widgets.text("output_dir", "/Volumes/workspace/default/semana3", "Carpeta de salida (gráficas)")
    BASE_PATH = dbutils.widgets.get("base_path").rstrip("/")
    OUTPUT_DIR = dbutils.widgets.get("output_dir").rstrip("/")
    EN_DATABRICKS = True
except Exception:
    BASE_PATH = "data"
    OUTPUT_DIR = "."
    EN_DATABRICKS = False

print(f"Origen de datos: {BASE_PATH}  ·  Databricks: {EN_DATABRICKS}")


def cargar_csv(nombre):
    """Lee un CSV tanto en Databricks (DBFS/Volumes) como en local."""
    candidatos = []
    if BASE_PATH.startswith("dbfs:/"):
        candidatos.append(BASE_PATH.replace("dbfs:/", "/dbfs/", 1) + "/" + nombre)
    candidatos.append(BASE_PATH + "/" + nombre)
    for ruta in candidatos:
        if os.path.exists(ruta):
            import pandas as pd
            return pd.read_csv(ruta)
    # Fallback Databricks sin FUSE: leer con Spark y pasar a pandas
    try:
        return spark.read.csv(f"{BASE_PATH}/{nombre}", header=True, inferSchema=True).toPandas()
    except Exception as e:
        raise FileNotFoundError(
            f"No encontré '{nombre}'. Verifica que esté en {BASE_PATH} "
            f"o ajusta el widget 'base_path'. Detalle: {e}"
        )

In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = cargar_csv("uplift_campaign.csv")
print(f"{len(df):,} clientes · {(df['treatment_group']=='treatment').sum():,} tratados · {(df['treatment_group']=='control').sum():,} control")
df.head()

## Paso 1 — entrenen un modelo de churn y úsenlo para targetear

Un modelo de churn normal: no sabe nada de tratamiento/control, solo aprende a predecir quién
**NO** va a volver a comprar. Lo usamos para elegir a quién contactar (targeting por riesgo) y
medimos qué tan bien o mal nos fue, usando el experimento real como juez.

In [0]:
FEATURES = ["recency_days", "frequency", "monetary", "email_open_rate", "tenure_days"]

train, test = train_test_split(df, test_size=0.30, random_state=42, stratify=df["treatment_group"])

train_churn = train.copy()
train_churn["churned"] = 1 - train_churn["responded_60d"]

# --- SOLUCIÓN: modelo de riesgo/churn entrenado SOLO con train ---
modelo_riesgo = LogisticRegression(max_iter=1000)
modelo_riesgo.fit(train_churn[FEATURES], train_churn["churned"])

test = test.copy()
test["prob_churn"] = modelo_riesgo.predict_proba(test[FEATURES])[:, 1]

print("Modelo de riesgo entrenado")
print("Coeficientes:", pd.Series(modelo_riesgo.coef_[0], index=FEATURES).round(3).to_dict())

In [0]:
def valor_de_la_politica(test_df, seleccionados):
    # Valor NETO: margen bruto menos el costo del cupón ($5, solo si compró) y el envío
    # ($0.05, por cualquier tratado) -- la misma fórmula de la slide "El experimento".
    sel = test_df.loc[seleccionados]
    valor_tratado = sel.loc[sel["treatment_group"] == "treatment", "utilidad_neta_60d"].mean()
    valor_control = sel.loc[sel["treatment_group"] == "control", "margin_60d"].mean()
    return valor_tratado - valor_control, len(sel)


n_contactar = int(len(test) * 0.30)

# --- Política por RIESGO (30% con mayor prob. de churn) ---
idx_riesgo = test.sort_values("prob_churn", ascending=False).head(n_contactar).index
uplift_riesgo, n_riesgo = valor_de_la_politica(test, idx_riesgo)
print(f"Targeting por riesgo: ${uplift_riesgo:+.2f}/cliente sobre {n_riesgo:,} clientes (total ${uplift_riesgo*n_riesgo:+,.2f})")

# --- Referencia: contactar al azar ---
idx_azar = test.sample(n=n_contactar, random_state=42).index
val_azar, _ = valor_de_la_politica(test, idx_azar)
print(f"Targeting al azar:    ${val_azar:+.2f}/cliente (total ${val_azar*n_contactar:+,.2f})")

**Lectura:** targetear por riesgo apenas queda por encima del azar (≈ +$0.07 vs −$1.56 por
cliente). El motivo: el churn mide *quién se va*, no *a quién el cupón convence*. Muchos de los
"en riesgo" son **Lost Cause** (no vuelven ni con cupón) y el cupón les regala margen.

## Paso 2 — programen el T-learner y midan su Qini

Dos regresiones logísticas: una entrenada SOLO con tratados, otra SOLO con control. Se les
pregunta a las dos sobre todo el conjunto de prueba y se restan.

In [0]:
train_t = train[train["treatment_group"] == "treatment"]
train_c = train[train["treatment_group"] == "control"]

# --- SOLUCIÓN: T-learner ---
modelo_tratado = LogisticRegression(max_iter=1000)
modelo_control = LogisticRegression(max_iter=1000)
modelo_tratado.fit(train_t[FEATURES], train_t["responded_60d"])
modelo_control.fit(train_c[FEATURES], train_c["responded_60d"])

X_test = test[FEATURES]
test["p_tratado"] = modelo_tratado.predict_proba(X_test)[:, 1]
test["p_control"] = modelo_control.predict_proba(X_test)[:, 1]
test["uplift_estimado"] = test["p_tratado"] - test["p_control"]

print("T-learner entrenado y conjunto de prueba puntuado")

In [0]:
def curva_qini(test_df, score_col, steps=40):
    orden = test_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    es_tratado = (orden["treatment_group"] == "treatment").values
    respondio = orden["responded_60d"].values

    acumulado_tratados = np.cumsum(np.where(es_tratado, respondio, 0))
    acumulado_control = np.cumsum(np.where(~es_tratado, respondio, 0))
    n_tratados_acum = np.cumsum(es_tratado)
    n_control_acum = np.cumsum(~es_tratado)

    n = len(orden)
    xs, ys = [0.0], [0.0]
    for i in np.linspace(1, n, steps).astype(int):
        nt, nc = n_tratados_acum[i - 1], n_control_acum[i - 1]
        tasa_t = acumulado_tratados[i - 1] / nt if nt > 0 else 0
        tasa_c = acumulado_control[i - 1] / nc if nc > 0 else 0
        xs.append(i / n)
        ys.append((tasa_t - tasa_c) * n)
    return np.array(xs), np.array(ys)


# Compatibilidad numpy 1.x / 2.x
try:
    _trap = np.trapezoid
except AttributeError:
    _trap = np.trapz


def coeficiente_qini(xs, ys, n):
    area_modelo = _trap(ys, xs)
    area_azar = _trap(np.linspace(0, ys[-1], len(xs)), xs)
    return (area_modelo - area_azar) / n


xs_uplift, ys_uplift = curva_qini(test, "uplift_estimado")
xs_riesgo, ys_riesgo = curva_qini(test, "prob_churn")
qini_uplift = coeficiente_qini(xs_uplift, ys_uplift, len(test))
qini_riesgo = coeficiente_qini(xs_riesgo, ys_riesgo, len(test))

print(f"Coeficiente Qini · T-learner:       {qini_uplift:+.3f}")
print(f"Coeficiente Qini · riesgo/churn:    {qini_riesgo:+.3f}")

In [0]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(xs_uplift, ys_uplift, color="#00704A", linewidth=3, label=f"T-learner (Qini={qini_uplift:+.3f})")
ax.plot(xs_riesgo, ys_riesgo, color="#C4512F", linewidth=3, label=f"Riesgo/churn (Qini={qini_riesgo:+.3f})")
ax.plot([0, 1], [0, ys_uplift[-1]], color="#9A988E", linestyle="--", label="Al azar")
ax.set_xlabel("% de la base contactada (de prueba)")
ax.set_ylabel("Respuestas incrementales acumuladas")
ax.set_title("Laboratorio Final — la curva Qini de su T-learner")
ax.legend()
plt.tight_layout()
try:
    plt.savefig(f"{OUTPUT_DIR}/curva_qini_laboratorio.png", dpi=150)
except Exception as e:
    print("No se pudo guardar la imagen:", e)
plt.show()

**Lectura:** el T-learner (Qini **+0.142**) ordena casi 3× mejor que el modelo de riesgo
(**+0.048**). Ambos superan el azar, pero el que decide bien *a quién le importa el cupón* es el
uplift, no el churn.

## Paso 3 — decidan a quién contactar

Probamos varios porcentajes de la base (10% … 100%) ordenados por `uplift_estimado` y, para cada
uno, calculamos el valor neto total.

In [0]:
resultados = []
for pct in range(10, 101, 10):
    n_c = int(len(test) * pct / 100)
    idx = test.sort_values("uplift_estimado", ascending=False).head(n_c).index
    valor_cliente, n = valor_de_la_politica(test, idx)
    resultados.append({"pct_contactado": pct, "valor_por_cliente": round(valor_cliente, 2),
                       "n_contactados": n, "valor_total": round(valor_cliente * n, 2)})

tabla_politicas = pd.DataFrame(resultados)
tabla_politicas

**Respuesta para defender (con nuestras cifras):**

| % contactado | Valor por cliente | Valor total |
|---:|---:|---:|
| 10% | **+$1.62** (máximo) | +$1,948.93 |
| **20%** | +$0.99 | **+$2,368.51 (máximo)** |
| 30% | +$0.46 | +$1,673.33 |
| 40% en adelante | negativo | negativo |

- El **valor TOTAL** se maximiza contactando al **20%** de la base (+$2,368.51): sumamos más
  clientes rentables sin meternos todavía en los de uplift ~0 o negativo.
- El **valor POR CLIENTE** se maximiza al **10%** (+$1.62): solo los más "Persuadable".
- **No son el mismo** porque el valor total = valor_por_cliente × n. Al pasar de 10% a 20% el
  uplift promedio baja, pero el volumen adicional de clientes todavía rentables compensa. A
  partir del 20% empiezan a entrar **Sleeping Dog** (el cupón los activa en contra) y **Lost
  Cause**, que hunden el promedio y el total.

**Recomendación:** contactar al **20% de mayor uplift**. Si el objetivo fuera eficiencia de
presupuesto (poco capital disponible), el 10%.

## La revelación final

El dataset completo (`uplift_campaign_FULL.csv`) trae el **tipo real** de cada cliente y su
**uplift verdadero** — información que en la vida real nunca existe.

In [0]:
full = cargar_csv("uplift_campaign_FULL.csv")
full_test = full[full["customer_id"].isin(test["customer_id"])].merge(
    test[["customer_id", "uplift_estimado", "prob_churn"]], on="customer_id"
)

print("Composición real de la base de prueba:")
print((full_test["true_type"].value_counts(normalize=True) * 100).round(1))
print()
print("Correlación uplift ESTIMADO vs uplift VERDADERO:",
      round(full_test["uplift_estimado"].corr(full_test["true_uplift_prob"]), 3))
print("Correlación prob. churn vs uplift VERDADERO:",
      round(full_test["prob_churn"].corr(full_test["true_uplift_prob"]), 3))
print()
print("Uplift real promedio por tipo:")
print(full_test.groupby("true_type")["true_uplift_prob"].mean().round(3))
print()
print("Valor real de contacto promedio por tipo:")
print(full_test.groupby("true_type")["true_value_of_contact"].mean().round(2))

**Interpretación de la revelación:**

- Composición real: **Sure Thing 48.9% · Persuadable 22.4% · Lost Cause 21.5% · Sleeping Dog 7.2%**.
  Solo el **22%** de la base es realmente "Persuadable" (a quien el cupón mueve la aguja).
- El uplift estimado correlaciona **0.619** con el verdadero, mientras que la prob. de churn
  correlaciona apenas **0.227** → el T-learner captura señal real de persuabilidad; el churn, casi
  nada.
- Coincide con la política óptima: contactar ~20-25% ≈ el tamaño del segmento Persuadable + algo
  de Sure Thing. Empujar más allá mete Sleeping Dogs (uplift −0.15) y Lost Causes (≈ 0), que es
  exactamente donde la tabla de políticas se vuelve negativa.

## Punto 4 (entregable) — cambien el modelo base del T-learner

Comparamos la logística contra Árbol de decisión, Random Forest, Gradient Boosting y SVM (RBF).
Cuidamos el sobreajuste con `max_depth` / `min_samples_leaf`. Métricas: **Qini** y **correlación
del uplift estimado con el verdadero**.

In [0]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


def evaluar_t_learner(nombre, make_model, escala=False):
    if escala:
        mt = Pipeline([("sc", StandardScaler()), ("m", make_model())])
        mc = Pipeline([("sc", StandardScaler()), ("m", make_model())])
    else:
        mt, mc = make_model(), make_model()

    mt.fit(train_t[FEATURES], train_t["responded_60d"])
    mc.fit(train_c[FEATURES], train_c["responded_60d"])

    tmp = test.copy()
    tmp["u"] = mt.predict_proba(test[FEATURES])[:, 1] - mc.predict_proba(test[FEATURES])[:, 1]

    xs, ys = curva_qini(tmp, "u")
    q = coeficiente_qini(xs, ys, len(tmp))

    merged = full[full["customer_id"].isin(tmp["customer_id"])].merge(
        tmp[["customer_id", "u"]], on="customer_id")
    corr = merged["u"].corr(merged["true_uplift_prob"])
    return {"modelo": nombre, "qini_uplift": round(q, 3), "corr_uplift": round(corr, 3)}


comparacion = [
    evaluar_t_learner("Regresión logística", lambda: LogisticRegression(max_iter=1000)),
    evaluar_t_learner("Árbol de decisión",
                      lambda: DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42)),
    evaluar_t_learner("Random Forest",
                      lambda: RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=25,
                                                     random_state=42, n_jobs=-1)),
    evaluar_t_learner("Gradient Boosting",
                      lambda: GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                         learning_rate=0.05, random_state=42)),
    evaluar_t_learner("SVM (RBF)",
                      lambda: SVC(kernel="rbf", C=1.0, probability=True, random_state=42), escala=True),
]

tabla_comparacion = pd.DataFrame(comparacion).sort_values("qini_uplift", ascending=False).reset_index(drop=True)
tabla_comparacion

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
t = tabla_comparacion
axes[0].barh(t["modelo"], t["qini_uplift"], color="#00704A")
axes[0].set_title("Qini del T-learner por modelo")
axes[0].invert_yaxis()
axes[1].barh(t["modelo"], t["corr_uplift"], color="#C4512F")
axes[1].set_title("Correlación uplift estimado vs verdadero")
axes[1].invert_yaxis()
plt.tight_layout()
try:
    plt.savefig(f"{OUTPUT_DIR}/comparacion_modelos.png", dpi=150)
except Exception as e:
    print("No se pudo guardar la imagen:", e)
plt.show()

**Conclusión del Punto 4 (nuestros resultados):**

| Modelo | Qini uplift | Corr. uplift real |
|---|---:|---:|
| **Random Forest** | **+0.158** | **0.868** |
| Gradient Boosting | +0.156 | 0.838 |
| SVM (RBF) | +0.146 | 0.797 |
| Árbol de decisión | +0.142 | 0.776 |
| Regresión logística | +0.142 | 0.619 |

- El **Random Forest** (con `max_depth=6`, `min_samples_leaf=25`) es el que **más mejora**: sube el
  Qini de 0.142 → 0.158 y, sobre todo, casi cuadruplica la correlación con el uplift real
  (0.619 → **0.868**). Gradient Boosting queda muy cerca.
- La logística y el árbol simple empatan en Qini (0.142), pero el árbol correlaciona mejor el
  uplift real (0.776 vs 0.619): captura no-linealidades que la logística no.
- Los árboles **sin límites** sobreajustan y el uplift estimado se vuelve ruido; por eso fijamos
  `max_depth` y `min_samples_leaf`. La regularización es la que hace que la mejora sea real y no
  ilusoria.

**Entregable final:** el notebook corrido de punta a punta + esta comparación, y la conclusión de
una página recomendando contactar al **20% de mayor uplift** (valor total máximo, $2,368.51 en
prueba), reconociendo que el modelo quedó a **0.87 de correlación** de la verdad.